# CO₂ retrieval showcase figures

This notebook makes shareable figures for the five CO₂ case studies. The edited plume GeoJSON files are the authoritative masks: no segmentation is rerun here. For each scene, it exports a 2 × 2 figure with the full matched-filter enhancement, full instrument uncertainty, and cropped, plume-clipped enhancement and uncertainty over RGB. It also exports the two cropped clipped GeoTIFFs and a five-scene RGB-overlay contact sheet.

Colour limits are calculated independently for each scene to make plume structure legible. They are therefore visualisation limits, not a cross-scene quantitative comparison.

In [ ]:
from __future__ import annotations

from datetime import datetime
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.warp import Resampling, reproject
from rasterio.windows import Window, transform as window_transform


## Inputs

Set the `CASE_STUDIES_CO2_ROOT` environment variable (or replace the placeholder in the next cell) with the directory containing the case-study data. The `plumes_path` entries point to the GeoJSON layers edited in QGIS; replace a relative path only when saving a later manual revision.


In [ ]:
CASE_STUDIES_CO2_ROOT = Path(
    os.environ.get("CASE_STUDIES_CO2_ROOT", "/path/to/case_studies_co2")
).expanduser()

CASES = {
    "Korba — Tanager-1 — 2025-02-19": {
        "site": "Korba MegaPlant, India",
        "sensor": "Tanager-1",
        "mf_display_percentiles": (2.0, 99.7),
        "mf_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/Tanager_data/20250219_053251_31_4001/CO2_full-column/basic_radiance_hdf5__20250219_053251_31_4001_basic_radiance_hdf5_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/Tanager_data/20250219_053251_31_4001/CO2_full-column/basic_radiance_hdf5__20250219_053251_31_4001_basic_radiance_hdf5_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/Tanager_data/20250219_053251_31_4001/CO2_full-column/basic_radiance_hdf5__20250219_053251_31_4001_basic_radiance_hdf5_co2_RGB.tif",
        "plumes_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/Tanager_data/20250219_053251_31_4001/CO2_full-column/plume_segmentation_co2/2026-08-11T19-30-35/basic_radiance_hdf5__20250219_053251_31_4001_basic_radiance_hdf5_co2_MF_components.geojson",
    },
    "Korba — EnMAP — 2023-11-06": {
        "site": "Korba MegaPlant, India",
        "sensor": "EnMAP",
        "mf_display_percentiles": (2.0, 99.3),
        "mf_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/EnMAP_data/ENMAP01-____L1B-DT0000050482_20231106T054649Z_003_V010506_20260810T142824Z_output/L1B_DT0000050482_003_20231106T054649Z_20231106T054654Z_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/EnMAP_data/ENMAP01-____L1B-DT0000050482_20231106T054649Z_003_V010506_20260810T142824Z_output/L1B_DT0000050482_003_20231106T054649Z_20231106T054654Z_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/EnMAP_data/ENMAP01-____L1B-DT0000050482_20231106T054649Z_003_V010506_20260810T142824Z_output/L1B_DT0000050482_003_20231106T054649Z_20231106T054654Z_co2_RGB.tif",
        "plumes_path": CASE_STUDIES_CO2_ROOT / "Korba_Plant_case_study/EnMAP_data/ENMAP01-____L1B-DT0000050482_20231106T054649Z_003_V010506_20260810T142824Z_output/plume_segmentation_co2/2026-08-11T20-02-59/L1B_DT0000050482_003_20231106T054649Z_20231106T054654Z_co2_MF_components.geojson",
    },
    "Matla — EnMAP — 2023-10-05": {
        "site": "Matla, South Africa",
        "sensor": "EnMAP",
        "mf_display_percentiles": (2.0, 99.5),
        "mf_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-10-05_Matla_South_Africa/CO2_full-column/L1B-DT0000044547_20231005T084547Z_003/L1B_DT0000044547_003_20231005T084547Z_20231005T084552Z_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-10-05_Matla_South_Africa/CO2_full-column/L1B-DT0000044547_20231005T084547Z_003/L1B_DT0000044547_003_20231005T084547Z_20231005T084552Z_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-10-05_Matla_South_Africa/CO2_full-column/L1B-DT0000044547_20231005T084547Z_003/L1B_DT0000044547_003_20231005T084547Z_20231005T084552Z_co2_RGB.tif",
        "plumes_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-10-05_Matla_South_Africa/CO2_full-column/L1B-DT0000044547_20231005T084547Z_003/plume_segmentation_co2/2026-08-11T19-06-15/L1B_DT0000044547_003_20231005T084547Z_20231005T084552Z_co2_MF_components.geojson",
    },
    "Riyadh PP09 — EnMAP — 2023-07-15": {
        "site": "Riyadh PP09, Saudi Arabia",
        "sensor": "EnMAP",
        "mf_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-15_PP09_Riyadh/CO2_full-column/L1B-DT0000027893_20230715T080658Z_001/L1B_DT0000027893_001_20230715T080658Z_20230715T080703Z_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-15_PP09_Riyadh/CO2_full-column/L1B-DT0000027893_20230715T080658Z_001/L1B_DT0000027893_001_20230715T080658Z_20230715T080703Z_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-15_PP09_Riyadh/CO2_full-column/L1B-DT0000027893_20230715T080658Z_001/L1B_DT0000027893_001_20230715T080658Z_20230715T080703Z_co2_RGB.tif",
        "plumes_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-15_PP09_Riyadh/CO2_full-column/L1B-DT0000027893_20230715T080658Z_001/plume_segmentation_co2/2026-08-11T19-06-37/L1B_DT0000027893_001_20230715T080658Z_20230715T080703Z_co2_MF_components.geojson",
    },
    "Riyadh PP10 — EnMAP — 2023-07-11": {
        "site": "Riyadh PP10, Saudi Arabia",
        "sensor": "EnMAP",
        "mf_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-11_PP10_Riyadh/CO2_full-column/L1B-DT0000027896_20230711T080336Z_001/L1B_DT0000027896_001_20230711T080336Z_20230711T080341Z_co2_MF.tif",
        "uncertainty_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-11_PP10_Riyadh/CO2_full-column/L1B-DT0000027896_20230711T080336Z_001/L1B_DT0000027896_001_20230711T080336Z_20230711T080341Z_co2_MF_uncertainty.tif",
        "rgb_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-11_PP10_Riyadh/CO2_full-column/L1B-DT0000027896_20230711T080336Z_001/L1B_DT0000027896_001_20230711T080336Z_20230711T080341Z_co2_RGB.tif",
        "plumes_path": CASE_STUDIES_CO2_ROOT / "enmap_search_aoi_Borger2025/2023-07-11_PP10_Riyadh/CO2_full-column/L1B-DT0000027896_20230711T080336Z_001/plume_segmentation_co2/2026-08-11T19-07-01/L1B_DT0000027896_001_20230711T080336Z_20230711T080341Z_co2_MF_components.geojson",
    },
}

ACTIVE_CASES = list(CASES)  # Use ["Korba — Tanager-1 — 2025-02-19"] to render only one scene.
RUN_STAMP = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
FIGURE_DPI = 240
PLUME_CROP_PADDING_PX = 12  # Padding on every side of the reviewed plume envelope.
MF_DISPLAY_PERCENTILES = (2.0, 99.99)  # Default; individual cases can override this.
UNCERTAINTY_DISPLAY_PERCENTILES = (2.0, 99.5)


In [ ]:
def require_files(case):
    for field in ("mf_path", "uncertainty_path", "rgb_path", "plumes_path"):
        if not case[field].is_file():
            raise FileNotFoundError(f"Missing {field}: {case[field]}")


def read_single_band(path):
    with rasterio.open(path) as src:
        array = src.read(1, out_dtype="float32")
        invalid = ~np.isfinite(array)
        if src.nodata is not None and np.isfinite(src.nodata):
            invalid |= array == src.nodata
        array[invalid] = np.nan
        return array, src.profile.copy(), src.transform, src.crs


def read_rgb_on_reference_grid(path, reference_profile):
    """Read RGB and resample it only when its grid differs from the MF grid."""
    with rasterio.open(path) as src:
        source = src.read(indexes=(1, 2, 3), out_dtype="float32")
        invalid = ~np.isfinite(source)
        if src.nodata is not None and np.isfinite(src.nodata):
            invalid |= source == src.nodata
        source[invalid] = np.nan
        same_grid = (
            src.width == reference_profile["width"]
            and src.height == reference_profile["height"]
            and src.transform == reference_profile["transform"]
            and src.crs == reference_profile["crs"]
        )
        if same_grid:
            return np.moveaxis(source, 0, -1)
        destination = np.full((3, reference_profile["height"], reference_profile["width"]), np.nan, dtype=np.float32)
        for band in range(3):
            reproject(
                source=source[band], destination=destination[band],
                src_transform=src.transform, src_crs=src.crs, src_nodata=np.nan,
                dst_transform=reference_profile["transform"], dst_crs=reference_profile["crs"], dst_nodata=np.nan,
                resampling=Resampling.bilinear,
            )
    return np.moveaxis(destination, 0, -1)


def stretch_rgb(rgb):
    result = np.zeros_like(rgb, dtype=np.float32)
    for band in range(3):
        values = rgb[..., band][np.isfinite(rgb[..., band])]
        if values.size == 0:
            continue
        low, high = np.percentile(values, (2, 98))
        if high > low:
            result[..., band] = np.clip((rgb[..., band] - low) / (high - low), 0, 1)
    result[~np.isfinite(rgb).all(axis=-1)] = np.nan
    return result


def reviewed_mask(plumes_path, shape, transform, crs):
    plumes = gpd.read_file(plumes_path)
    if plumes.crs is None:
        raise ValueError(f"The edited plume layer has no CRS: {plumes_path}")
    plumes = plumes.to_crs(crs)
    plumes = plumes[plumes.geometry.notna() & ~plumes.geometry.is_empty]
    if plumes.empty:
        raise ValueError(f"The edited plume layer contains no valid polygons: {plumes_path}")
    mask = rasterize(
        ((geometry, 1) for geometry in plumes.geometry),
        out_shape=shape, transform=transform, fill=0, dtype=np.uint8, all_touched=False,
    ).astype(bool)
    return mask, plumes


def colour_limits(array, lower=2, upper=99.5):
    values = array[np.isfinite(array)]
    if values.size == 0:
        raise ValueError("Cannot determine plotting limits from an all-nodata array.")
    vmin, vmax = np.percentile(values, (lower, upper))
    return (float(vmin), float(vmax)) if vmax > vmin else (float(vmin - 1), float(vmax + 1))


def write_clipped_raster(path, array, profile):
    out_profile = profile.copy()
    out_profile.update(count=1, dtype=rasterio.float32, nodata=np.nan, compress="lzw")
    with rasterio.open(path, "w", **out_profile) as dst:
        dst.write(array.astype(np.float32), 1)


def plume_window(mask, padding_px):
    """Return a square raster window around the reviewed plume envelope plus padding."""
    rows, columns = np.where(mask)
    if rows.size == 0:
        raise ValueError("Cannot crop an empty plume mask.")
    plume_height = int(rows.max() - rows.min() + 1)
    plume_width = int(columns.max() - columns.min() + 1)
    side = min(max(plume_height, plume_width) + 2 * padding_px, min(mask.shape))
    row_centre = (rows.min() + rows.max() + 1) / 2
    col_centre = (columns.min() + columns.max() + 1) / 2
    row_start = int(round(row_centre - side / 2))
    col_start = int(round(col_centre - side / 2))
    row_start = min(max(0, row_start), mask.shape[0] - side)
    col_start = min(max(0, col_start), mask.shape[1] - side)
    return Window(col_start, row_start, side, side)


def window_slices(window):
    return (
        slice(int(window.row_off), int(window.row_off + window.height)),
        slice(int(window.col_off), int(window.col_off + window.width)),
    )


def cropped_profile(profile, window):
    result = profile.copy()
    result.update(
        height=int(window.height), width=int(window.width),
        transform=window_transform(window, profile["transform"]),
    )
    return result


def add_colourbar(fig, image, axis, label):
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03, label=label)


def plot_case(case_name, case, mf, uncertainty, rgb, mask, crop_window, output_path):
    mf_percentiles = case.get("mf_display_percentiles", MF_DISPLAY_PERCENTILES)
    mf_limits = colour_limits(mf, *mf_percentiles)
    uncertainty_limits = colour_limits(uncertainty, *UNCERTAINTY_DISPLAY_PERCENTILES)
    row_slice, col_slice = window_slices(crop_window)
    mf_crop = mf[row_slice, col_slice]
    uncertainty_crop = uncertainty[row_slice, col_slice]
    rgb_crop = rgb[row_slice, col_slice]
    mask_crop = mask[row_slice, col_slice]
    mf_clipped = np.where(mask_crop, mf_crop, np.nan)
    uncertainty_clipped = np.where(mask_crop, uncertainty_crop, np.nan)

    fig, axes = plt.subplots(2, 2, figsize=(15, 13), constrained_layout=True)
    image = axes[0, 0].imshow(mf, cmap="viridis", vmin=mf_limits[0], vmax=mf_limits[1])
    axes[0, 0].contour(mask, levels=[0.5], colors="white", linewidths=0.8)
    axes[0, 0].set_title("CO₂ matched-filter enhancement")
    add_colourbar(fig, image, axes[0, 0], "ΔX (ppm m)")

    image = axes[0, 1].imshow(uncertainty, cmap="magma", vmin=uncertainty_limits[0], vmax=uncertainty_limits[1])
    axes[0, 1].contour(mask, levels=[0.5], colors="white", linewidths=0.8)
    axes[0, 1].set_title("Instrument uncertainty")
    add_colourbar(fig, image, axes[0, 1], "σ_RMN (ppm m)")

    axes[1, 0].imshow(rgb_crop)
    image = axes[1, 0].imshow(np.ma.masked_invalid(mf_clipped), cmap="viridis", vmin=mf_limits[0], vmax=mf_limits[1], alpha=0.78)
    axes[1, 0].contour(mask_crop, levels=[0.5], colors="white", linewidths=1.0)
    axes[1, 0].set_title("Reviewed CO₂ plume enhancement over RGB (cropped)")
    add_colourbar(fig, image, axes[1, 0], "ΔX (ppm m; same scale as full map)")

    axes[1, 1].imshow(rgb_crop)
    image = axes[1, 1].imshow(np.ma.masked_invalid(uncertainty_clipped), cmap="magma", vmin=uncertainty_limits[0], vmax=uncertainty_limits[1], alpha=0.78)
    axes[1, 1].contour(mask_crop, levels=[0.5], colors="white", linewidths=1.0)
    axes[1, 1].set_title("Plume uncertainty over RGB (cropped)")
    add_colourbar(fig, image, axes[1, 1], "σ_RMN (ppm m; same scale as full map)")

    for axis in axes.ravel():
        axis.set_axis_off()
    fig.suptitle(f"{case_name}\n{case['site']}", fontsize=16, fontweight="bold")
    fig.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    return mf_clipped, uncertainty_clipped, mf_limits, uncertainty_limits, rgb_crop, mask_crop


def save_individual_overlay(item, value_key, limits_key, cmap, title, colourbar_label, output_path):
    """Save one square plume-over-RGB figure with its own colourbar."""
    figure, axis = plt.subplots(figsize=(7.5, 7.5), constrained_layout=True)
    axis.imshow(item["rgb"])
    limits = item[limits_key]
    image = axis.imshow(
        np.ma.masked_invalid(item[value_key]), cmap=cmap,
        vmin=limits[0], vmax=limits[1], alpha=0.78,
    )
    axis.contour(item["mask"], levels=[0.5], colors="white", linewidths=1.0)
    axis.set_title(f"{item['case_name']}\n{title}", fontsize=13, fontweight="bold")
    axis.set_axis_off()
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.03, label=colourbar_label)
    figure.savefig(output_path, dpi=FIGURE_DPI, facecolor="white")
    plt.show()
    plt.close(figure)


## Render and export

Outputs are written next to each matched-filter file under `co2_retrieval_showcase/<timestamp>/`. In addition to the four-panel QA figure and clipped GeoTIFFs, the last part of the cell exports two independent square images for every case: MF plume over RGB and plume uncertainty over RGB.

In [ ]:
unknown_cases = [case_name for case_name in ACTIVE_CASES if case_name not in CASES]
if unknown_cases:
    raise KeyError(f"Unknown case(s): {unknown_cases}")

rendered = []
for case_name in ACTIVE_CASES:
    case = CASES[case_name]
    require_files(case)
    mf, profile, transform, crs = read_single_band(case["mf_path"])
    uncertainty, uncertainty_profile, uncertainty_transform, uncertainty_crs = read_single_band(case["uncertainty_path"])
    if uncertainty.shape != mf.shape or uncertainty_transform != transform or uncertainty_crs != crs:
        raise ValueError(f"Uncertainty is not aligned to MF for {case_name}.")
    profile.update(transform=transform, crs=crs)
    mask, edited_polygons = reviewed_mask(case["plumes_path"], mf.shape, transform, crs)
    mask &= np.isfinite(mf)
    if not mask.any():
        raise ValueError(f"Edited polygons do not overlap valid MF pixels for {case_name}.")
    rgb = stretch_rgb(read_rgb_on_reference_grid(case["rgb_path"], profile))
    crop_window = plume_window(mask, PLUME_CROP_PADDING_PX)
    mf_crop_profile = cropped_profile(profile, crop_window)
    uncertainty_crop_profile = cropped_profile(uncertainty_profile, crop_window)

    output_dir = case["mf_path"].parent / "co2_retrieval_showcase" / RUN_STAMP
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = case["mf_path"].stem
    figure_path = output_dir / f"{stem}_showcase.png"
    mf_clip_path = output_dir / f"{stem}_reviewed_plume_MF.tif"
    uncertainty_clip_path = output_dir / f"{stem}_reviewed_plume_uncertainty.tif"

    mf_clipped, uncertainty_clipped, mf_limits, uncertainty_limits, rgb_crop, mask_crop = plot_case(
        case_name, case, mf, uncertainty, rgb, mask, crop_window, figure_path
    )
    write_clipped_raster(mf_clip_path, mf_clipped, mf_crop_profile)
    write_clipped_raster(uncertainty_clip_path, uncertainty_clipped, uncertainty_crop_profile)
    rendered.append({
        "case_name": case_name, "rgb": rgb_crop, "mask": mask_crop,
        "mf_clipped": mf_clipped, "mf_limits": mf_limits,
        "uncertainty_clipped": uncertainty_clipped, "uncertainty_limits": uncertainty_limits,
        "output_dir": output_dir, "stem": stem,
    })
    print(f"{case_name}: {len(edited_polygons)} reviewed polygon(s); {mask.sum():,} plume pixel(s)")
    print(f"  Figure: {figure_path}")
    print(f"  Clipped MF: {mf_clip_path}")
    print(f"  Clipped uncertainty: {uncertainty_clip_path}")

for item in rendered:
    mf_overlay_path = item["output_dir"] / f"{item['stem']}_MF_plume_over_RGB.png"
    uncertainty_overlay_path = item["output_dir"] / f"{item['stem']}_uncertainty_plume_over_RGB.png"
    save_individual_overlay(
        item, "mf_clipped", "mf_limits", "viridis",
        "Reviewed CO₂ plume enhancement over RGB",
        "ΔX (ppm m; same scale as full map)", mf_overlay_path,
    )
    save_individual_overlay(
        item, "uncertainty_clipped", "uncertainty_limits", "magma",
        "Reviewed plume uncertainty over RGB",
        "σ_RMN (ppm m; same scale as full map)", uncertainty_overlay_path,
    )
    print(f"{item['case_name']} independent overlays:")
    print(f"  MF over RGB: {mf_overlay_path}")
    print(f"  Uncertainty over RGB: {uncertainty_overlay_path}")


The outline and overlay use precisely the edited GeoJSON polygons. The clipped GeoTIFFs are square crops centred on the reviewed plume envelope plus `PLUME_CROP_PADDING_PX` and use `NaN` outside the reviewed plume, making them suitable inputs for the later CO₂ IME workflow.